#📓 Notebook Prático — Spark ML com a Base Real NYC TLC (Unidade 5)

## Ambiente e download dos dados reais

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Unidade5_ML_NYCTaxi_Real") \
    .getOrCreate()

print("Versão do Spark:", spark.version)

Versão do Spark: 4.0.4


In [2]:
!wget -q https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [3]:
from datasets import load_dataset

ds = load_dataset("alexvaroz/nyc_tripdata_2024_sample_100k", split="train")
df = spark.createDataFrame(ds.to_pandas())

part-00000-11784945-4f78-47c8-b6e2-d36cb(…): reconstructing file:   0%|          |  0.00B / 2.65MB            

part-00000-11784945-4f78-47c8-b6e2-d36cb(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [4]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
df.count()

100000

### Verificando o ambiente criado.

Qual a quantidade de executores no Colab? Qual o número de partições? O número de partições influencia alguma coisa??

In [ ]:
spark

In [75]:
sc = spark.sparkContext

In [76]:
print(sc.defaultParallelism)   # nº de núcleos usados = tarefas simultâneas

2


Número padrão de partições:

In [77]:
df.rdd.getNumPartitions()

2

### Experimento para checar o efeito do número de partições no processamento.

In [6]:
df_base = df.select('payment_type', 'total_amount').cache()

In [7]:
import time
from pyspark.sql import functions as F

def consulta(d):
    return d.groupBy('payment_type').agg(
        F.count('*').alias('corridas'),
        F.sum('total_amount').alias('receita')
    )

def medir(n_part, repeticoes=3):
    '''Executa a agregação com n_part partições de shuffle e devolve a mediana do tempo.'''
    spark.conf.set('spark.sql.shuffle.partitions', str(n_part))
    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = consulta(df_base)
        resultado.collect()  # ação: dispara a execução
        tempos.append(time.perf_counter() - inicio)
    mediana = sorted(tempos)[len(tempos) // 2]
    return mediana, resultado.rdd.getNumPartitions()

In [83]:
import pandas as pd

valores = [1, 2, 4, 8, 16, 50, 200, 1000]
resultados = []
for n in valores:
    t, p = medir(n)
    resultados.append({'shuffle_partitions': n, 'tempo_s': round(t, 3), 'particoes_resultado': p})
    print(f'{n:>5} partições -> {t:.3f} s')

tabela = pd.DataFrame(resultados)
tabela

    1 partições -> 0.213 s
    2 partições -> 0.172 s
    4 partições -> 0.259 s
    8 partições -> 0.163 s
   16 partições -> 0.232 s
   50 partições -> 0.593 s
  200 partições -> 0.953 s
 1000 partições -> 3.863 s


,shuffle_partitions,tempo_s,particoes_resultado
0,1,0.213,1
1,2,0.172,2
2,4,0.259,4
3,8,0.163,8
4,16,0.232,16
5,50,0.593,50
6,200,0.953,200
7,1000,3.863,1000


## Plotando o resultado em um gráfico

In [8]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(tabela['shuffle_partitions'], tabela['tempo_s'], marker='o')
plt.xscale('log')
plt.xticks(valores, [str(v) for v in valores])
plt.axvline(sc.defaultParallelism, linestyle='--', color='gray', label=f'nº de núcleos ({sc.defaultParallelism})')
plt.xlabel('spark.sql.shuffle.partitions (escala log)')
plt.ylabel('Tempo da agregação (s)')
plt.title('Tempo do groupBy x número de partições de shuffle')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

NameError: name 'tabela' is not defined

<Figure size 800x400 with 0 Axes>

Vamos fazer uso de 4 partições então:

In [62]:
spark.conf.set('spark.sql.shuffle.partitions', str(4))

In [66]:
sc.defaultParallelism

2

In [68]:
df.rdd.getNumPartitions()

2

In [67]:
from pyspark.sql.functions import spark_partition_id

df.groupBy(spark_partition_id().alias('particao')).count().orderBy('particao').show()

+--------+-----+
|particao|count|
+--------+-----+
|       0|50176|
|       1|49824|
+--------+-----+



In [69]:
print(df.groupBy('payment_type').count().rdd.getNumPartitions())  # este sim é afetado

1


In [70]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
print('AQE desligado:', df.groupBy('payment_type').count().rdd.getNumPartitions())  # 4


AQE desligado: 4


In [71]:
df.rdd.getNumPartitions()

2

In [73]:
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .config("spark.executor.cores", "1")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

In [74]:
spark

### Limpeza dos nulos nas colunas

In [ ]:
from pyspark.sql.functions import col, sum as _sum

df.select("trip_distance", "fare_amount", "total_amount",
          "passenger_count").describe().show()



+-------+------------------+-----------------+------------------+---------------+
|summary|     trip_distance|      fare_amount|      total_amount|passenger_count|
+-------+------------------+-----------------+------------------+---------------+
|  count|            100000|           100000|            100000|         100000|
|   mean|3.3597048000000043| 19.2431963999999|27.793334399999296|            NaN|
| stddev| 4.421642813040028|20.01269857810465|24.583842633586812|            NaN|
|    min|               0.0|           -523.1|           -550.73|            0.0|
|    max|             86.27|          1852.32|           1852.32|            NaN|
+-------+------------------+-----------------+------------------+---------------+



In [25]:
# Nulos por coluna
cols_chave = ["passenger_count", "trip_distance", "fare_amount",
              "payment_type", "tpep_pickup_datetime"]
df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in cols_chave]).show()

+---------------+-------------+-----------+------------+--------------------+
|passenger_count|trip_distance|fare_amount|payment_type|tpep_pickup_datetime|
+---------------+-------------+-----------+------------+--------------------+
|              0|            0|          0|           0|                   0|
+---------------+-------------+-----------+------------+--------------------+



Testes de consistência lógica: regras que o mundo real impõe — o desembarque não pode ocorrer antes do embarque; a tarifa mínima oficial de um táxi amarelo é $2,50

In [26]:
total = df.count()

problemas = {
    "Distância zero ou negativa": df.filter(col("trip_distance") <= 0).count(),
    "Tarifa negativa ou zero":    df.filter(col("fare_amount") <= 0).count(),
    "Tarifas absurdas (> $300)":  df.filter(col("fare_amount") > 300).count(),
    "Passageiros = 0":            df.filter(col("passenger_count") == 0).count(),
    "Dropoff antes do pickup":    df.filter(
        col("tpep_dropoff_datetime") < col("tpep_pickup_datetime")).count(),
}

for nome, qtd in problemas.items():
    print(f"{nome}: {qtd:,} ({qtd/total*100:.2f}%)")

Distância zero ou negativa: 1,816 (1.82%)
Tarifa negativa ou zero: 1,780 (1.78%)
Tarifas absurdas (> $300): 13 (0.01%)
Passageiros = 0: 961 (0.96%)
Dropoff antes do pickup: 6 (0.01%)


In [27]:
df_limpo = (df
    .filter(col("trip_distance") > 0)
    .filter(col("fare_amount") >= 2.50)              # tarifa mínima oficial
    .filter(col("fare_amount") <= 300)               # remove casos extremos
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    .filter(col("payment_type").isin([1, 2]))        # apenas cartão e dinheiro
)

print(f"Original: {total:,} | Após filtros: {df_limpo.count():,}")

Original: 100,000 | Após filtros: 86,057


Outliers via IQR. O intervalo interquartil define limites estatísticos para valores "aceitáveis".

Registros fora desse intervalo são candidatos a remoção — mas a decisão final é de negócio, não apenas estatística.

In [ ]:
q1, q3 = df_limpo.approxQuantile("fare_amount", [0.25, 0.75], 0.01)
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Limites aceitáveis: [{lim_inf:.2f}, {lim_sup:.2f}]")

#df_limpo = df_limpo.filter(
#    (col("fare_amount") >= lim_inf) & (col("fare_amount") <= lim_sup))

Limites aceitáveis: [-8.55, 39.05]


In [ ]:
df.filter(col("fare_amount") >= 39.05).count()

10657

In [34]:
df.filter(col("passenger_count") != 0).select(sum("passenger_count")).show()

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [ ]:
from pyspark.sql.functions import median

# Calcula e exibe a média de passageiros
df.select(mean("passenger_count").alias("media_passageiros")).show()

In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(inputCols=["passenger_count"],
                 outputCols=["passengers"]).setStrategy("median")
df_limpo = imputer.fit(df_limpo).transform(df_limpo)

In [ ]:
antes = df_limpo.count()
df_limpo = df_limpo.dropDuplicates()
print(f"Duplicatas exatas removidas: {antes - df_limpo.count():,}")

df_limpo.cache()  # evita recomputação nas próximas atividades

Duplicatas exatas removidas: 0


DataFrame[VendorID: bigint, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: bigint, DOLocationID: bigint, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, passengers: double]

A qualidade de um modelo depende menos do algoritmo e mais dos atributos que o alimentam. Nesta etapa, extraímos estrutura preditiva dos campos brutos:

* Atributos temporais: hora do dia e dia da semana capturam padrões de demanda (picos, madrugadas, fins de semana) que influenciam tarifa e duração.
* Duração da corrida: calculada como a diferença entre os timestamps de embarque e desembarque, é frequentemente o preditores mais forte da tarifa — o trânsito de NYC faz duas corridas de mesma distância valerem diferente. Esse atributo só existe porque combinamos duas colunas.
* Custo por milha: razões entre variáveis expõem relações que o valor absoluto esconde.
Integração de fontes (join). O PULocationID é apenas um código numérico. O join com a tabela taxi_zone_lookup traduz cada código em bairro (Borough), enriquecendo o dataset — exatamente o cenário real de combinar fontes distintas por uma chave de ligação.

Divisão treino/teste. Separamos 80% para treino e 20% para teste com randomSplit. O conjunto de treino ensina o modelo; o de teste avalia sua capacidade de generalizar para dados nunca vistos. A semente (seed=42) garante reprodutibilidade: divisões idênticas em execuções diferentes.

In [ ]:
from pyspark.sql.functions import (hour, dayofweek, when, round as _round,
                                   unix_timestamp)

df_feat = (df_limpo
    .withColumn("trip_duration_min",
        _round((unix_timestamp("tpep_dropoff_datetime")
              - unix_timestamp("tpep_pickup_datetime")) / 60, 2))
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", dayofweek("tpep_pickup_datetime"))
    .withColumn("eh_fim_de_semana",
        when(dayofweek("tpep_pickup_datetime").isin([1, 7]), 1.0).otherwise(0.0))
    .withColumn("custo_por_milha",
        _round(col("fare_amount") / col("trip_distance"), 2))
)

df_feat.select("trip_distance", "trip_duration_min",
               "fare_amount").show(5)

In [ ]:
zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
zonas.select("LocationID", "Borough", "Zone").show(5)

df_feat = (df_feat
    .join(zonas.select("LocationID", "Borough")
           .withColumnRenamed("Borough", "pickup_borough"),
          df_feat.PULocationID == zonas.LocationID, "left")
    .drop("LocationID")
)

df_feat.groupBy("pickup_borough").count().orderBy("count", ascending=False).show()

In [ ]:
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)
print(f"Treino: {train_df.count():,} | Teste: {test_df.count():,}")

In [ ]:
train_df.printSchema()

In [ ]:
from pyspark.sql.functions import create_map, lit, col
from itertools import chain
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder,
                                VectorAssembler, StandardScaler)

# Mapeamento dos códigos oficiais da TLC
mapa_pagamento = {1: "credito", 2: "dinheiro"}
mapping_expr = create_map(*[lit(x) for x in
    chain.from_iterable(mapa_pagamento.items())])

# Adiciona a coluna ao df_feat
df_feat = df_feat.withColumn(
    "payment_label", mapping_expr[col("payment_type").cast("int")])

# Refaz o split para garantir que train_df e test_df tenham a nova coluna
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)

num_cols = ["trip_distance", "trip_duration_min", "passengers",
            "pickup_hour", "eh_fim_de_semana"]

indexer = StringIndexer(inputCol="payment_label", outputCol="payment_idx",
                        handleInvalid="skip")
encoder = OneHotEncoder(inputCol="payment_idx", outputCol="payment_vec")
assembler_num = VectorAssembler(inputCols=num_cols, outputCol="features_num")
scaler = StandardScaler(inputCol="features_num",
                        outputCol="features_num_scaled",
                        withMean=True, withStd=True)
assembler_final = VectorAssembler(
    inputCols=["features_num_scaled", "payment_vec"], outputCol="features")

pipeline = Pipeline(stages=[indexer, encoder, assembler_num,
                            scaler, assembler_final])
pipeline_model = pipeline.fit(train_df)   # fit APENAS no treino

train_prep = pipeline_model.transform(train_df)
test_prep  = pipeline_model.transform(test_df)

In [ ]:
pipeline_model.write().overwrite().save("/tmp/pipeline_nyc_real")

In [ ]:
from pyspark.ml import PipelineModel
PipelineModel.load("/tmp/pipeline_nyc_real") \
    .transform(test_df.limit(5)) \
    .select("features", "fare_amount").show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import when

mediana = df_feat.approxQuantile("fare_amount", [0.5], 0.01)[0]
print(f"Tarifa mediana real: ${mediana:.2f}")

df_cls = df_feat.withColumn(
    "label", when(col("fare_amount") > mediana, 1.0).otherwise(0.0))

train_c, test_c = df_cls.randomSplit([0.8, 0.2], seed=42)
train_c_prep = pipeline_model.transform(train_c)
test_c_prep  = pipeline_model.transform(test_c)

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

modelo_lr = LogisticRegression(featuresCol="features", labelCol="label",
                               maxIter=1000).fit(train_c_prep)

In [ ]:
modelo_rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                                   numTrees=50, seed=42).fit(train_c_prep)

In [ ]:
pred_lr = modelo_lr.transform(test_c_prep)
pred_rf = modelo_rf.transform(test_c_prep)

In [ ]:
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator)

evs = {
    "AUC":      BinaryClassificationEvaluator(labelCol="label",
                                              metricName="areaUnderROC"),
    "F1":       MulticlassClassificationEvaluator(labelCol="label",
                                                  metricName="f1"),
    "Precisão": MulticlassClassificationEvaluator(labelCol="label",
                                                  metricName="weightedPrecision"),
    "Recall":   MulticlassClassificationEvaluator(labelCol="label",
                                                  metricName="weightedRecall"),
}

for nome, pred in [("Logistic Regression", pred_lr),
                   ("Random Forest", pred_rf)]:
    print(f"{nome}: " + " | ".join(
        f"{m}={e.evaluate(pred):.3f}" for m, e in evs.items()))

In [ ]:
# Executar se tiver tempo... É bem demorado
"""
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import numpy as np

lr_base = LogisticRegression(featuresCol="features", labelCol="label")
grid = (ParamGridBuilder()
        .addGrid(lr_base.regParam, [0.01, 0.1, 1.0])
        .addGrid(lr_base.maxIter, [10, 50])
        .build())

cv = CrossValidator(estimator=lr_base, estimatorParamMaps=grid,
                    evaluator=BinaryClassificationEvaluator(labelCol="label"),
                    numFolds=3, parallelism=4, seed=42)
cv_model = cv.fit(train_c_prep)

melhor_idx = int(np.argmax(cv_model.avgMetrics))
print("Melhores parâmetros:", cv.getEstimatorParamMaps()[melhor_idx])
print(f"Melhor AUC médio: {max(cv_model.avgMetrics):.3f}")
"""

# Regressão

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr_reg = LinearRegression(featuresCol="features", labelCol="fare_amount")
pred_reg = lr_reg.fit(train_prep).transform(test_prep)

for metrica in ["rmse", "mae", "r2"]:
    ev = RegressionEvaluator(labelCol="fare_amount",
                             predictionCol="prediction",
                             metricName=metrica)
    print(f"{metrica.upper():4}: {ev.evaluate(pred_reg):.3f}")

pred_reg.select("fare_amount", "prediction").show(10)

In [ ]:
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor

ev = RegressionEvaluator(labelCol="fare_amount",
                         predictionCol="prediction", metricName="rmse")

rf_reg = RandomForestRegressor(featuresCol="features",
                               labelCol="fare_amount",
                               numTrees=50, seed=42).fit(train_prep)
print(f"Random Forest RMSE: {ev.evaluate(rf_reg.transform(test_prep)):.3f}")

In [ ]:
gbt = GBTRegressor(featuresCol="features", labelCol="fare_amount",
                   maxIter=50, seed=42).fit(train_prep)
print(f"GBT RMSE: {ev.evaluate(gbt.transform(test_prep)):.3f}")

In [ ]:
# Quais atributos mais influenciam a tarifa? (pergunta do material)
import pandas as pd
imp = pd.DataFrame({"feature": num_cols + ["payment_vec"],
                    "importance": rf_reg.featureImportances.toArray()})
print(imp.sort_values("importance", ascending=False))

# Clusterização

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

assembler_km = VectorAssembler(
    inputCols=["trip_distance", "trip_duration_min", "fare_amount", "passengers"],
    outputCol="features_km_raw")
scaler_km = StandardScaler(inputCol="features_km_raw",
                           outputCol="features", withMean=True, withStd=True)

df_km = Pipeline(stages=[assembler_km, scaler_km]) \
    .fit(df_feat).transform(df_feat)

modelo_km = KMeans(featuresCol="features", predictionCol="cluster",
                   k=4, seed=42).fit(df_km)
resultado = modelo_km.transform(df_km)

In [ ]:
avaliador = ClusteringEvaluator(featuresCol="features")

for k in range(2, 9):
    m = KMeans(k=k, seed=42, featuresCol="features").fit(df_km)
    sil = avaliador.evaluate(m.transform(df_km))
    print(f"K={k} | Custo={m.summary.trainingCost:14.2f} | Silhouette={sil:.4f}")

# Perfil médio de cada cluster em valores originais
resultado.groupBy("cluster").avg(
    "trip_distance", "trip_duration_min", "fare_amount").show()